# DeepWeeds Day 2 — Ho Ngoc Mai (02509)

Reproducible Kaggle notebook. Run cells in order. Use **validation only** for every model, hyperparameter and inference choice. The final cell is deliberately separate: after it is run, do not revise the configuration from test metrics.


In [ ]:
# 1. Install and fetch the submitted main branch.
!pip -q install timm thop
!git clone --depth 1 https://github.com/ngmai2005/K4-Track4-Day2-Deeplearning-Advance.git /kaggle/working/lab02509 || git -C /kaggle/working/lab02509 pull --ff-only
%cd /kaggle/working/lab02509/submissions/02509_ho_ngoc_mai
import sys, platform, torch, timm
sys.path[:0] = ['code', '/kaggle/working/lab02509']
assert torch.cuda.is_available(), 'Enable a Kaggle GPU session before running experiments.'
print({'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'gpu': torch.cuda.get_device_name(0)})


In [ ]:
# 2. Download DeepWeeds, verify MD5, and fetch canonical fold-0 CSV files.
from pathlib import Path
import hashlib, urllib.request, zipfile
DATA = Path('data'); LABELS = DATA / 'labels'
DATA.mkdir(exist_ok=True); LABELS.mkdir(exist_ok=True)
archive = DATA / 'images.zip'
if not archive.exists():
    urllib.request.urlretrieve('https://zenodo.org/records/7939060/files/images.zip?download=1', archive)
h = hashlib.md5()
with archive.open('rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
assert h.hexdigest() == 'b7b30f96d466fba86016aa5a26606e0f', h.hexdigest()
if len(list(DATA.glob('*.jpg'))) != 17509:
    with zipfile.ZipFile(archive) as z: z.extractall(DATA)
for name in ('labels', 'train_subset0', 'val_subset0', 'test_subset0'):
    target = LABELS / f'{name}.csv'
    if not target.exists():
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels/{name}.csv', target)
IMAGES_DIR = DATA
print({'md5': h.hexdigest(), 'images': len(list(IMAGES_DIR.glob('*.jpg'))), 'label_csv': sorted(p.name for p in LABELS.glob('*.csv'))})


In [ ]:
# 3. Mandatory split and implementation checks.
from checks import main as run_checks
from dataset import load_split, check_split
run_checks()
tr, va, te = load_split(LABELS, fold=0)
split_stats = check_split(tr, va, te, IMAGES_DIR)
print(split_stats)


In [ ]:
# 4. EDA: save the fold-0 class distribution used in the report.
import matplotlib.pyplot as plt
from pathlib import Path
Path('figures').mkdir(exist_ok=True)
counts = {name: frame.groupby('Species').size() for name, frame in {'train': tr, 'val': va, 'test': te}.items()}
ax = __import__('pandas').DataFrame(counts).plot.bar(figsize=(12, 4))
ax.set_ylabel('Images'); ax.set_title('DeepWeeds fold-0 class distribution'); ax.figure.tight_layout()
ax.figure.savefig('figures/eda_class_distribution.png', dpi=160)
plt.show()
print('largest/smallest train ratio:', counts['train'].max() / counts['train'].min())


In [ ]:
# 5. Sanity checks before costly runs: forward/backward and fixed-batch overfit.
import numpy as np
from dataset import build_transforms, make_loader
from model import build_model
from losses import build_criterion
loader = make_loader(tr.iloc[:32], IMAGES_DIR, build_transforms(train=True, img_size=224), 8, True, num_workers=0)
x, y, _ = next(iter(loader)); x, y = x.cuda(), y.cuda()
model = build_model('mobilenetv3_large_100', pretrained=True).cuda(); criterion = build_criterion('ce')
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
losses = []
for _ in range(25):
    logits = model(x[:8]); loss = criterion(logits, y[:8])
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step(); losses.append(float(loss))
assert np.isfinite(losses).all() and losses[-1] < losses[0], losses
print({'initial_loss': losses[0], 'final_fixed_batch_loss': losses[-1], 'expected_uniform_ce': float(np.log(9))})


In [ ]:
# 6. Backbone screen: exactly the same T00 recipe and seed 0; validation only.
from train import Config, run
from experiment_plan import BACKBONES
common = dict(images_dir=str(IMAGES_DIR), labels_dir=str(LABELS), out_dir='runs', pred_dir='predictions', epochs=12, batch_size=64)
backbone_results = [run(Config(**{**common, **cfg.__dict__})) for cfg in BACKBONES]
backbone_results


## Decision gate
Read the validation-only `backbone_results`, latency measurements and curves. Set `SELECTED_BACKBONE` below before continuing. Do not use the test set at this stage.


In [ ]:
# 7. Controlled training ablations. Only change one declared factor from T00 at a time.
from experiment_plan import TRAINING
SELECTED_BACKBONE = 'resnet50'  # replace after the validation-only backbone decision
training_results = []
for cfg in TRAINING:
    cfg_dict = {**common, **cfg.__dict__, 'backbone': SELECTED_BACKBONE}
    training_results.append(run(Config(**cfg_dict)))
training_results


## Final decision gate
Use only `training_results`, validation logits and validation latency to write the locked `FINAL_CONFIG` below. Then run the final cell once. It writes one test-prediction CSV per seed for the final configuration and the T00/I00 baseline.


In [ ]:
# 8. Inference evaluation on validation only. Add I00 plus at least four methods here.
# Required methods: single view, hflip/multicrop TTA, probability/logit aggregation,
# temperature scaling fitted on val, and FP16 or Conv-BN fusion where applicable.
# Record p50/p95/p99 at batch 1 and batch 32 with benchmark.latency_report.


In [ ]:
# 9. FINAL — run only once after completing the validation decision gates above.
# FINAL_CONFIG must be copied from the selected validation configuration; do not tune it after this cell.
FINAL_CONFIG = dict(backbone=SELECTED_BACKBONE, aug='basic', loss='ce', mix=None, ema_decay=None)
final_results = []
for seed in (0, 1, 2):
    final_results.append(run(Config(exp_id='F01', seed=seed, save_test_predictions=True, **common, **FINAL_CONFIG)))
    final_results.append(run(Config(exp_id='T00', seed=seed, save_test_predictions=True, **common, backbone=SELECTED_BACKBONE)))
final_results
# Then use eval.py score/grade to produce eval_out/ before filling results.xlsx and report.md.
